# 장시간 실행 에이전트를 위한 컨텍스트 편집과 메모리

여러 세션에 걸쳐 실행되거나 오래 걸리는 작업을 다루는 AI 에이전트에는 두 가지 핵심 난제가 있습니다. 대화가 바뀌면 학습한 패턴을 잃어버린다는 것, 그리고 긴 상호작용 중에 컨텍스트 윈도가 가득 찬다는 것입니다.

이 쿡북에서는 Claude의 메모리 도구와 컨텍스트 편집 기능으로 이 난제들을 해결하는 방법을 보여 줍니다.

## 목차

1. [들어가며: 메모리가 중요한 이유](#introduction)
2. [활용 사례](#use-cases)
3. [빠른 시작 예제](#quick-start)
4. [동작 방식](#how-it-works)
5. [코드 리뷰 어시스턴트 데모](#demo)
6. [실제 적용 사례](#real-world)
7. [모범 사례](#best-practices)

## 사전 준비

**필요한 사전 지식:**
- Python 기초(함수, 클래스, async/await 기본)
- REST API와 JSON에 대한 기본 이해

**필요한 도구:**
- Python 3.10 이상
- Anthropic API 키 ([여기서 발급](https://console.anthropic.com/))

**권장:**
- 동시성 프로그래밍 개념(스레드, async)에 대한 친숙함
- LLM의 컨텍스트 윈도에 대한 기본 이해

## 준비

### VSCode 사용자

```bash
# 1. Create virtual environment
python -m venv .venv

# 2. Activate it
source .venv/bin/activate  # macOS/Linux
# or: .venv\Scripts\activate  # Windows

# 3. Install dependencies
pip install -r requirements.txt

# 4. In VSCode: Select .venv as kernel (top right)
```

### API 키

```bash
cp .env.example .env
# Edit .env and add your ANTHROPIC_API_KEY
```

API 키는 https://console.anthropic.com/ 에서 발급받으세요.

## 1. 들어가며: 메모리가 중요한 이유 {#introduction}

이 쿡북은 [AI 에이전트를 위한 효과적인 컨텍스트 엔지니어링](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents)에서 설명한 컨텍스트 엔지니어링 패턴을 실제로 구현해 봅니다. 해당 글은 컨텍스트가 왜 유한한 자원인지, 어텐션 예산이 어떻게 동작하는지, 효과적인 에이전트를 만드는 전략은 무엇인지를 다룹니다. 여기서는 그 기법들을 직접 실행해 봅니다.

### 문제

대규모 언어 모델의 컨텍스트 윈도는 유한합니다(Claude 4 계열 모델은 200k 토큰). 커 보이지만 다음과 같은 문제들이 생깁니다.

- **컨텍스트 한도**: 긴 대화나 복잡한 작업은 가용 컨텍스트를 초과할 수 있습니다
- **연산 비용**: 큰 컨텍스트를 처리하는 것은 비쌉니다. 어텐션 메커니즘은 제곱으로 증가합니다
- **반복되는 패턴**: 대화가 바뀔 때마다 비슷한 작업의 맥락을 매번 다시 설명해야 합니다
- **정보 손실**: 컨텍스트가 가득 차면 앞부분의 중요한 정보가 사라집니다

### 해결책

Claude 4 모델은 강력한 컨텍스트 관리 기능을 제공합니다.

1. **메모리 도구**(`memory_20250818`): 대화를 넘나드는 학습을 가능하게 합니다
   - Claude가 배운 것을 나중에 참고하려고 기록해 둘 수 있습니다
   - `/memories` 디렉터리 아래의 파일 기반 시스템입니다
   - 클라이언트 측 구현이라 여러분이 완전히 제어할 수 있습니다

2. **컨텍스트 편집**: 두 가지 전략으로 컨텍스트를 자동 관리합니다
   - **도구 사용 정리**(`clear_tool_uses_20250919`): 컨텍스트가 커지면 오래된 도구 결과를 지웁니다
   - **사고 관리**(`clear_thinking_20251015`): 확장 사고 블록을 관리합니다(사고 기능 활성화 필요)
   - 촉발 조건과 보존 정책을 설정할 수 있습니다

### 이점

**시간이 지날수록 여러분의 작업을 더 잘 해내는** AI 에이전트를 만들 수 있습니다.

- **세션 1**: Claude가 문제를 풀고 그 패턴을 기록해 둡니다
- **세션 2**: Claude가 학습한 패턴을 곧바로 적용합니다(더 빠릅니다!)
- **긴 세션**: 컨텍스트 편집이 대화를 감당 가능한 크기로 유지합니다

사람이 그러듯 Claude에게 메모하고 다시 꺼내 볼 수 있는 노트를 쥐여 주는 셈입니다.

### 배울 내용

이 쿡북을 마치면 다음을 할 수 있습니다.
- 대화를 넘나드는 학습을 위해 메모리 도구를 **구현**하기
- 장시간 세션을 관리하도록 컨텍스트 편집을 **설정**하기
- 메모리 보안과 구성에 대한 모범 사례를 **적용**하기

## 1. 들어가며: 메모리가 중요한 이유 {#introduction}

이 쿡북은 [AI 에이전트를 위한 효과적인 컨텍스트 엔지니어링](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents)에서 설명한 컨텍스트 엔지니어링 패턴을 실제로 구현해 봅니다. 해당 글은 컨텍스트가 왜 유한한 자원인지, 어텐션 예산이 어떻게 동작하는지, 효과적인 에이전트를 만드는 전략은 무엇인지를 다룹니다. 여기서는 그 기법들을 직접 실행해 봅니다.

### 문제

대규모 언어 모델의 컨텍스트 윈도는 유한합니다(Claude 4는 200k 토큰). 커 보이지만 다음과 같은 문제들이 생깁니다.

- **컨텍스트 한도**: 긴 대화나 복잡한 작업은 가용 컨텍스트를 초과할 수 있습니다
- **연산 비용**: 큰 컨텍스트를 처리하는 것은 비쌉니다. 어텐션 메커니즘은 제곱으로 증가합니다
- **반복되는 패턴**: 대화가 바뀔 때마다 비슷한 작업의 맥락을 매번 다시 설명해야 합니다
- **정보 손실**: 컨텍스트가 가득 차면 앞부분의 중요한 정보가 사라집니다

### 해결책

Claude Sonnet 4.6은 두 가지 강력한 기능을 제공합니다.

1. **메모리 도구**(`memory_20250818`): 대화를 넘나드는 학습을 가능하게 합니다
   - Claude가 배운 것을 나중에 참고하려고 기록해 둘 수 있습니다
   - `/memories` 디렉터리 아래의 파일 기반 시스템입니다
   - 클라이언트 측 구현이라 여러분이 완전히 제어할 수 있습니다

**지원 모델**: Claude Opus 4.1(`claude-opus-4-1`), Claude Opus 4(`claude-opus-4`), Claude Sonnet 4.6(`claude-sonnet-4-6`), Claude Sonnet 4(`claude-sonnet-4`), Claude Haiku 4.5(`claude-haiku-4-5`)

### 이점

**시간이 지날수록 여러분의 작업을 더 잘 해내는** AI 에이전트를 만들 수 있습니다.

- **세션 1**: Claude가 문제를 풀고 그 패턴을 기록해 둡니다
- **세션 2**: Claude가 학습한 패턴을 곧바로 적용합니다(더 빠릅니다!)
- **긴 세션**: 컨텍스트 편집이 대화를 감당 가능한 크기로 유지합니다

사람이 그러듯 Claude에게 메모하고 다시 꺼내 볼 수 있는 노트를 쥐여 주는 셈입니다.

## 2. 활용 사례 {#use-cases}

메모리와 컨텍스트 관리는 강력한 새 워크플로를 가능하게 합니다.

### 🔍 코드 리뷰 어시스턴트
- 지난 리뷰에서 디버깅 패턴을 학습합니다
- 이후 세션에서 비슷한 버그를 즉시 알아봅니다
- 팀 고유의 코드 품질 지식을 쌓아 갑니다
- **프로덕션 적용 가능**: GitHub PR 리뷰에 [claude-code-action](https://github.com/anthropics/claude-code-action)과 연동하세요

### 📚 리서치 어시스턴트
- 여러 세션에 걸쳐 주제별 지식을 축적합니다
- 서로 다른 연구 갈래의 통찰을 연결합니다
- 참고 문헌과 출처 추적을 유지합니다

### 💬 고객 지원 봇
- 사용자의 선호와 소통 방식을 학습합니다
- 자주 발생하는 문제와 해결책을 기억합니다
- 상호작용을 통해 제품 지식 베이스를 만들어 갑니다

### 📊 데이터 분석 도우미
- 데이터셋의 패턴과 이상치를 기억합니다
- 잘 통했던 분석 기법을 저장합니다
- 시간이 지나며 도메인 특화 통찰을 쌓아 갑니다

**지원 모델**: Claude Opus 4.1(`claude-opus-4-1`), Claude Sonnet 4.6(`claude-sonnet-4-6`)

**이 쿡북은 코드 리뷰 어시스턴트에 집중합니다.** 메모리(패턴 학습)와 컨텍스트 편집(긴 리뷰 처리)을 모두 잘 보여 주기 때문입니다.

## 3. 빠른 시작 예제 {#quick-start}

간단한 예제로 메모리와 컨텍스트 관리가 실제로 동작하는 모습을 살펴보겠습니다.

### 준비

먼저 의존성을 설치하고 환경을 설정합니다:

In [1]:
%%capture
# Install required packages
# Option 1: From requirements.txt
# %pip install -q -r requirements.txt

# Option 2: Direct install
%pip install -q anthropic python-dotenv ipykernel

**⚠️ 중요**: 이 디렉터리에 `.env` 파일을 만드세요:

```bash
# Copy .env.example to .env and add your API key
cp .env.example .env
```

그런 다음 `.env`를 편집해 https://console.anthropic.com/ 에서 발급받은 Anthropic API 키를 추가하세요.

In [2]:
import os
from typing import cast

from anthropic import Anthropic
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Model configuration - use alias for automatic updates
MODEL = "claude-sonnet-4-6"  # Can override via ANTHROPIC_MODEL env var
if os.getenv("ANTHROPIC_MODEL"):
    MODEL = os.getenv("ANTHROPIC_MODEL")

if not API_KEY:
    raise ValueError("ANTHROPIC_API_KEY not found. Copy .env.example to .env and add your API key.")

if not MODEL:
    raise ValueError("ANTHROPIC_MODEL not found. Copy .env.example to .env and set the model.")

MODEL = cast(str, MODEL)

client = Anthropic(api_key=API_KEY)

print("✓ API key loaded")
print(f"✓ Using model: {MODEL}")

✓ API key loaded
✓ Using model: claude-sonnet-4-6


### 예제 1: 기본 메모리 사용

Claude가 나중에 참고할 정보를 메모리에 저장하는 모습을 살펴보겠습니다.

**헬퍼 함수**

이 예제들은 `demo_helpers.py`의 헬퍼 함수를 사용합니다.

- **`run_conversation_loop()`**: API 대화 루프를 처리합니다
  - 메모리 도구를 활성화한 채 Claude API를 호출합니다
  - 도구 사용(메모리 작업)을 실행합니다
  - Claude가 도구 사용을 멈출 때까지 이어 갑니다
  - 최종 응답을 반환합니다

- **`run_conversation_turn()`**: 단일 턴(예제 3에서 사용)
  - 위와 같지만 API 호출 한 번 뒤에 반환합니다
  - 세밀한 제어가 필요할 때 유용합니다

- **`print_context_management_info()`**: 컨텍스트 정리 통계를 표시합니다
  - 절약한 토큰 수, 정리된 도구 사용 수를 보여 줍니다
  - 컨텍스트 편집이 언제 촉발되는지 파악하는 데 도움이 됩니다

**⚠️ 메모리 삭제에 대한 참고**

다음 셀은 이 시연을 깨끗한 상태에서 시작하기 위해 모든 메모리 파일을 삭제합니다. 노트북을 여러 번 실행하며 일관된 결과를 보고 싶을 때 유용합니다.

**실제 애플리케이션에서는** 모든 메모리를 삭제할지 신중히 판단해야 합니다. 학습한 패턴이 영구히 사라지기 때문입니다. 대신 선택적 삭제를 하거나 메모리를 프로젝트별 디렉터리로 나누는 방법을 고려하세요.

In [3]:
# Import helper functions
from memory_demo.demo_helpers import (
    run_conversation_loop,
    run_conversation_turn,
    print_context_management_info,
)
from memory_tool import MemoryToolHandler

# Initialize
client = Anthropic()
memory = MemoryToolHandler(base_path="./demo_memory")

# Clear any existing memories to start fresh
print("🧹 Clearing previous memories...")
memory.clear_all_memory()
print("✓ Memory cleared\n")

# Load example code with a race condition bug
with open("memory_demo/sample_code/web_scraper_v1.py", "r") as f:
    code_to_review = f.read()

messages = [
    {
        "role": "user",
        "content": f"I'm reviewing a multi-threaded web scraper that sometimes returns fewer results than expected. The count is inconsistent across runs. Can you find the issue?\n\n```python\n{code_to_review}\n```",
    }
]

print("=" * 60)
print("📝 SESSION 1: Learning from a bug")
print("=" * 60)

# Run conversation loop
response = run_conversation_loop(
    client=client,
    model=MODEL,
    messages=messages,
    memory_handler=memory,
    system="You are a code reviewer.",
    max_tokens=2048,
    max_turns=5,
    verbose=True,
)

print("\n" + "=" * 60)
print("✅ Session 1 complete!")
print("=" * 60)

🧹 Clearing previous memories...
✓ Memory cleared

📝 SESSION 1: Learning from a bug

🔄 Turn 1:
💬 Claude: I'll review this code for the race condition issue. Let me first check my memory, then analyze the problem.

  🔧 Memory tool: view /memories
  ✓ Result: Directory: /memories
(empty)

🔄 Turn 2:
  🔧 Memory tool: create /memories/review.md
  ✓ Result: File created successfully at /memories/review.md

🔄 Turn 3:
💬 Claude: ## Code Review: Multi-threaded Web Scraper Race Condition

### 🔴 **Critical Issue Found: Race Condition in Shared State**

You've correctly identified the problem! The code has **race conditions** caused by multiple threads modifying shared lists without synchronization.

---

### **The Problem**

**Lines with race conditions:**
```python
self.results.append(result)  # RACE CONDITION
```

**Problem:** 
- Python's `list.append()` is **NOT thread-safe** for concurrent modifications
- Multiple threads simultaneously appending can cause:
  - **Lost updates:** One thread's ap

**무슨 일이 일어났나요?**

1. Claude가 메모리를 확인했습니다(첫 실행이라 비어 있음)
2. 버그를 찾아냈습니다: **경쟁 상태(race condition)** — 여러 스레드가 동기화 없이 공유 상태(`self.results`와 `self.failed_urls`)를 수정하고 있었습니다
3. 나중에 참고하도록 동시성 패턴을 메모리에 저장했습니다

이제 진짜 흥미로운 부분입니다. Claude가 **새 대화**에서 학습한 패턴을 적용하는 모습을 보겠습니다:

### 예제 2: 대화를 넘나드는 학습

완전히 새로운 대화를 시작합니다. 메모리는 그대로 남아 있습니다!

In [4]:
# NEW conversation (empty messages)
# Load API client code with similar concurrency issue
with open("memory_demo/sample_code/api_client_v1.py", "r") as f:
    code_to_review = f.read()

messages = [
    {"role": "user", "content": f"Review this API client code:\n\n```python\n{code_to_review}\n```"}
]

print("=" * 60)
print("🚀 SESSION 2: Applying learned pattern")
print("=" * 60)

# Run conversation loop
response = run_conversation_loop(
    client=client,
    model=MODEL,
    messages=messages,
    memory_handler=memory,
    system="You are a code reviewer.",
    max_tokens=2048,
    max_turns=5,
    verbose=True,
)

print("\n" + "=" * 60)
print("✅ Session 2 complete!")
print("=" * 60)

🚀 SESSION 2: Applying learned pattern

🔄 Turn 1:
  🔧 Memory tool: view /memories
  ✓ Result: Directory: /memories
- review.md

🔄 Turn 2:
  🔧 Memory tool: view /memories/review.md
  ✓ Result:    1: # Code Review: Multi-threaded Web Scraper
   2: 
   3: ## Issue
   4: Revi...

🔄 Turn 3:
  🔧 Memory tool: str_replace /memories/review.md
  ✓ Result: File /memories/review.md has been edited successfully

🔄 Turn 4:
💬 Claude: ## Code Review: Async API Client

### Summary
This code has **concurrency issues** related to shared mutable state being accessed from multiple coroutines without proper synchronization. While the author correctly identifies these as bugs in comments, let me provide a comprehensive review.

However, the actual race condition risk here is **somewhat overstated** because:
- List `.append()` is atomic in CPython
- The `+=` operation on integers is also atomic
- Coroutines only switch at `await` points, and there are none between the operations

**BUT** this is still problema

**차이를 눈여겨보세요:**

- Claude가 **곧바로 메모리를 확인해** 스레드 안전성/동시성 패턴을 찾아냈습니다
- 다시 학습할 필요 없이 비동기 코드에서 비슷한 문제를 **즉시** 알아봤습니다
- 공유 가변 상태에 대해 저장해 둔 지식을 적용했기에 응답이 **더 빨랐습니다**

이것이 바로 **대화를 넘나드는 학습**입니다!

### 예제 3: 메모리는 보존하면서 컨텍스트 정리하기

여러 코드 파일을 다루는 **긴 리뷰 세션**에서는 어떤 일이 벌어질까요?

- 이전 리뷰의 도구 결과로 컨텍스트가 가득 찹니다
- 하지만 메모리(학습한 패턴)는 반드시 유지되어야 합니다!

**컨텍스트 편집**을 촉발해 Claude가 이를 어떻게 자동으로 관리하는지 살펴보겠습니다.

**설정에 대한 참고:** 여기서는 `clear_at_least: 50` 토큰을 사용합니다. 메모리 도구 작업은 결과가 작기 때문입니다(각각 약 50~150 토큰). 웹 검색이나 코드 실행처럼 도구 결과가 큰 실제 환경에서는 3000~5000 토큰 같은 더 큰 값을 쓰게 됩니다.

In [5]:
# Configure context management with BOTH clearing strategies
# Low thresholds for demo - in production use 30-40k tokens
CONTEXT_MANAGEMENT = {
    "edits": [
        # Thinking management MUST come first when combining strategies
        {
            "type": "clear_thinking_20251015",
            "keep": {"type": "thinking_turns", "value": 1}  # Keep only last turn's thinking
        },
        {
            "type": "clear_tool_uses_20250919",
            "trigger": {"type": "input_tokens", "value": 5000},  # Low threshold for demo
            "keep": {"type": "tool_uses", "value": 2},  # Keep last 2 tool uses
            "clear_at_least": {"type": "input_tokens", "value": 2000}
        }
    ]
}

# Extended thinking config (required for clear_thinking strategy)
THINKING = {
    "type": "enabled",
    "budget_tokens": 1024  # Budget for thinking per turn
}

# Continue from previous session - memory persists!
print("=" * 60)
print("📚 SESSION 3: Long review session with context clearing")
print("=" * 60)
print()

# Clean up messages - remove any empty content from previous session
# This ensures we have a valid message state to continue from
cleaned_messages = []
for msg in messages:
    if isinstance(msg.get("content"), list):
        # Filter out empty content blocks
        content = [c for c in msg["content"] if c]
        if content:
            cleaned_messages.append({"role": msg["role"], "content": content})
    elif msg.get("content"):
        cleaned_messages.append(msg)

messages = cleaned_messages

# Review 1: Data processor (larger file)
with open("memory_demo/sample_code/data_processor_v1.py", "r") as f:
    data_processor_code = f.read()

messages.append({
    "role": "user",
    "content": f"Review this data processor:\n\n```python\n{data_processor_code}\n```"
})

print("📝 Review 1: Data processor")
response = run_conversation_turn(
    client=client,
    model=MODEL,
    messages=messages,
    memory_handler=memory,
    system="You are a code reviewer.",
    context_management=CONTEXT_MANAGEMENT,
    thinking=THINKING,
    max_tokens=4096,
    verbose=True
)

# Add response to messages
messages.append({"role": "assistant", "content": response[1]})
if response[2]:
    messages.append({"role": "user", "content": response[2]})

print(f"  📊 Input tokens: {response[0].usage.input_tokens:,}")
context_cleared, saved = print_context_management_info(response[0])
print()

# Review 2: SQL code
with open("memory_demo/sample_code/sql_query_builder.py", "r") as f:
    sql_code = f.read()

messages.append({
    "role": "user",
    "content": f"Review this SQL query builder:\n\n```python\n{sql_code}\n```"
})

print("📝 Review 2: SQL query builder")
response = run_conversation_turn(
    client=client,
    model=MODEL,
    messages=messages,
    memory_handler=memory,
    system="You are a code reviewer.",
    context_management=CONTEXT_MANAGEMENT,
    thinking=THINKING,
    max_tokens=4096,
    verbose=True
)

messages.append({"role": "assistant", "content": response[1]})
if response[2]:
    messages.append({"role": "user", "content": response[2]})

print(f"  📊 Input tokens: {response[0].usage.input_tokens:,}")
context_cleared, saved = print_context_management_info(response[0])
print()

# Review 3: Add one more review to ensure we trigger clearing
with open("memory_demo/sample_code/web_scraper_v1.py", "r") as f:
    scraper_code = f.read()

messages.append({
    "role": "user",
    "content": f"Quick check - any issues here?\n\n```python\n{scraper_code}\n```"
})

print("📝 Review 3: Web scraper (should trigger clearing)")
response = run_conversation_turn(
    client=client,
    model=MODEL,
    messages=messages,
    memory_handler=memory,
    system="You are a code reviewer.",
    context_management=CONTEXT_MANAGEMENT,
    thinking=THINKING,
    max_tokens=4096,
    verbose=True
)

messages.append({"role": "assistant", "content": response[1]})
if response[2]:
    messages.append({"role": "user", "content": response[2]})

print(f"  📊 Input tokens: {response[0].usage.input_tokens:,}")
context_cleared, saved = print_context_management_info(response[0])

print()
print("=" * 60)
print("✅ Session 3 complete!")
print("=" * 60)

📚 SESSION 3: Long review session with context clearing

📝 Review 1: Data processor
🧠 Thinking: The user wants me to review a data processor with multiple concurrency and thread-safety issues. Let...
  🔧 Memory tool: str_replace /memories/review.md
  ✓ Result: File /memories/review.md has been edited successfully
  📊 Input tokens: 6,611
  ℹ️  Context below threshold - no clearing triggered

📝 Review 2: SQL query builder
🧠 Thinking: The user is now asking me to review a SQL query builder with SQL injection vulnerabilities. This is ...
  🔧 Memory tool: str_replace /memories/review.md
  ✓ Result: File /memories/review.md has been edited successfully
  📊 Input tokens: 7,923
  ✂️  Context editing triggered!
      • Cleared 1 thinking turn(s), saved 166 tokens
      • After clearing: 7,923 tokens

📝 Review 3: Web scraper (should trigger clearing)
🧠 Thinking: This is a quick check request for a web scraper with threading issues. Let me quickly identify the p...
  🔧 Memory tool: str_replace /me

**방금 무슨 일이 일어났나요?**

확장 사고를 켠 채로 여러 리뷰를 진행하며 컨텍스트가 커지자 컨텍스트 편집이 적용되었습니다.
1. **사고 블록 정리** — 이전 턴의 오래된 사고가 먼저 제거되었습니다
2. **도구 결과 정리** — 임계치를 넘자 오래된 메모리 도구 결과가 제거되었습니다
3. **메모리 파일은 그대로** — Claude는 여전히 학습한 패턴을 조회할 수 있습니다
4. **토큰 사용량 관리** — 사고와 도구 결과 양쪽에서 토큰을 절약했습니다

여기서 핵심 이점이 드러납니다.
- **단기 기억**(대화 컨텍스트 + 사고) → 공간을 확보하기 위해 정리됨
- **장기 기억**(저장된 패턴) → 세션을 넘어 유지됨

정리 이후에도 메모리가 살아남았는지 확인해 보겠습니다:

In [6]:
# Verify memory persists after context clearing
import os

print("📂 Memory files in demo_memory/:")
print()

for root, dirs, files in os.walk("./demo_memory"):
    # Calculate relative path for display
    level = root.replace("./demo_memory", "").count(os.sep)
    indent = "  " * level
    folder_name = os.path.basename(root) or "demo_memory"
    print(f"{indent}{folder_name}/")

    sub_indent = "  " * (level + 1)
    for file in files:
        file_path = os.path.join(root, file)
        size = os.path.getsize(file_path)
        print(f"{sub_indent}├── {file} ({size} bytes)")

print()
print("✅ All learned patterns preserved despite context clearing!")

📂 Memory files in demo_memory/:

demo_memory/
  memories/
    ├── review.md (318 bytes)

✅ All learned patterns preserved despite context clearing!


## 4. 동작 방식 {#how-it-works}

### 메모리 도구 아키텍처

메모리 도구는 **클라이언트 측**에서 동작합니다. 저장은 여러분이 제어합니다. Claude가 도구를 호출하면 여러분의 애플리케이션이 그것을 실행합니다.

#### 메모리 도구 명령

| 명령 | 설명 | 예시 |
|---------|-------------|---------|
| `view` | 디렉터리 또는 파일 내용 보기 | `{"command": "view", "path": "/memories"}` |
| `create` | 파일 생성 또는 덮어쓰기 | `{"command": "create", "path": "/memories/notes.md", "file_text": "..."}` |
| `str_replace` | 파일 안의 텍스트 치환 | `{"command": "str_replace", "path": "...", "old_str": "...", "new_str": "..."}` |
| `insert` | 지정한 줄 번호에 텍스트 삽입 | `{"command": "insert", "path": "...", "insert_line": 2, "insert_text": "..."}` |
| `delete` | 파일 또는 디렉터리 삭제 | `{"command": "delete", "path": "/memories/old.txt"}` |
| `rename` | 파일 이름 변경 또는 이동 | `{"command": "rename", "old_path": "...", "new_path": "..."}` |

경로 검증과 보안 조치를 포함한 전체 구현은 `memory_tool.py`를 참고하세요.

### 사고 관리(`clear_thinking_20251015`)

확장 사고를 사용하면 사고 블록이 쌓이면서 토큰을 소비합니다. `clear_thinking` 전략이 이를 자동으로 관리해 줍니다.

**중요**: 이 전략은 API 호출에서 `thinking`이 활성화되어 있어야 합니다.

**API 호출 패턴**(확장 사고 활성화):

```python
response = client.beta.messages.create(
    betas=["context-management-2025-06-27"],  # Required beta flag
    model="claude-sonnet-4-6",
    messages=messages,
    tools=[{"type": "memory_20250818", "name": "memory"}],
    thinking={"type": "enabled", "budget_tokens": 10000},  # Enable thinking
    context_management={  # Context editing config
        "edits": [
            {
                "type": "clear_thinking_20251015",
                "keep": {"type": "thinking_turns", "value": 1}  # Keep last turn only
            },
            {
                "type": "clear_tool_uses_20250919",
                "trigger": {"type": "input_tokens", "value": 35000},
                "keep": {"type": "tool_uses", "value": 5}
            }
        ]
    },
    max_tokens=2048
)
```

**핵심 사항:**
- 전략을 함께 쓸 때는 `clear_thinking`이 **먼저** 와야 합니다
- 확장 사고가 활성화되어 있어야 합니다(`thinking={"type": "enabled", ...}`)
- 캐시 적중을 극대화하려면 `"keep": "all"`로 모든 사고 블록을 보존하세요
- 사고에 대해서는 트리거가 선택 사항입니다(`keep` 값에 따라 정리됩니다)

### 데모 코드 이해하기

`code_review_demo.py`의 핵심 구현 내용입니다:

```python
class CodeReviewAssistant:
    def __init__(self, memory_storage_path="./memory_storage"):
        self.client = Anthropic()
        self.memory_handler = MemoryToolHandler(base_path=memory_storage_path)
        self.messages = []
    
    def review_code(self, code, filename, description=""):
        # 1. Add user message
        self.messages.append({...})
        
        # 2. Conversation loop with tool execution
        while True:
            response = self.client.beta.messages.create(
                model=MODEL,
                system=self._create_system_prompt(),
                messages=self.messages,
                tools=[{"type": "memory_20250818", "name": "memory"}],
                betas=["context-management-2025-06-27"],
                context_management=CONTEXT_MANAGEMENT
            )
            
            # 3. Execute tool uses
            tool_results = []
            for content in response.content:
                if content.type == "tool_use":
                    result = self._execute_tool_use(content)
                    tool_results.append({...})
            
            # 4. Continue if there are tool uses, otherwise done
            if tool_results:
                self.messages.append({"role": "user", "content": tool_results})
            else:
                break
```

**핵심 패턴**: 도구 사용이 있는 동안 계속 API를 호출하면서, 도구를 실행하고 그 결과를 다시 돌려주는 것입니다.

### Claude가 실제로 학습하는 것

메모리를 강력하게 만드는 것은 단순한 문법이 아니라 **의미 수준의 패턴 인식**입니다.

**세션 1: 스레드 기반 웹 스크레이퍼**

```python
# Bug: Race condition
class WebScraper:
    def __init__(self):
        self.results = []  # Shared state!
    
    def scrape_urls(self, urls):
        with ThreadPoolExecutor() as executor:
            for future in as_completed(futures):
                self.results.append(future.result())  # RACE!
```

**Claude가 메모리에 저장하는 것**(예시 파일: `/memories/concurrency_patterns/thread_safety.md`):

Claude는 이 패턴을 마주하면 다음과 같은 통찰을 메모리 파일에 저장합니다.
- **증상**: 동시 작업에서 결과가 일관되지 않음
- **원인**: 여러 스레드에서 수정되는 공유 가변 상태(리스트/딕셔너리)
- **해결책**: 락, 스레드 안전 자료구조를 쓰거나 결과를 반환하는 방식으로 변경
- **위험 신호**: 스레드 콜백 안의 인스턴스 변수, 사용되지 않는 락, 카운터 증가

---

**세션 2: 비동기 API 클라이언트**(새 대화!)

Claude는 먼저 메모리를 확인해 스레드 안전성 패턴을 찾은 뒤 다음을 수행합니다.
1. 비동기 코드에서도 비슷한 패턴을 **알아봅니다**(코루틴도 서로 끼어들 수 있습니다)
2. 해결책을 곧바로 **적용합니다**(다시 학습할 필요 없음)
3. 저장된 지식을 근거로 **설명합니다**

```python
# Claude spots this immediately:
async def fetch_all(self, endpoints):
    for coro in asyncio.as_completed(tasks):
        self.responses.append(await coro)  # Same pattern!
```

---

**이것이 중요한 이유:**

- ❌ **문법 검사기**는 경쟁 상태를 전혀 잡아내지 못합니다
- ✅ **Claude는** 아키텍처 수준의 패턴을 학습해 여러 맥락에 적용합니다
- ✅ **언어를 넘나듭니다**: Go, Java, Rust의 동시성에도 같은 패턴이 적용됩니다
- ✅ **점점 나아집니다**: 리뷰할 때마다 지식 베이스가 쌓입니다

### 샘플 코드 파일

이 데모는 다음 샘플 파일을 사용합니다(모두 동시성/스레드 안전성 버그를 갖고 있습니다).

- `memory_demo/sample_code/web_scraper_v1.py` — 경쟁 상태: 여러 스레드가 공유 상태를 수정
- `memory_demo/sample_code/api_client_v1.py` — 비동기 맥락의 유사한 동시성 버그
- `memory_demo/sample_code/data_processor_v1.py` — 긴 세션 데모를 위한 복수의 동시성 문제

하나를 살펴보겠습니다:

**`memory_demo/sample_code/web_scraper_v1.py`**

```python
"""
Concurrent web scraper with a race condition bug.
Multiple threads modify shared state without synchronization.
"""

import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List, Dict

import requests


class WebScraper:
    """Web scraper that fetches multiple URLs concurrently."""

    def __init__(self, max_workers: int = 10):
        self.max_workers = max_workers
        self.results = []  # BUG: Shared mutable state accessed by multiple threads!
        self.failed_urls = []  # BUG: Another race condition!

    def fetch_url(self, url: str) -> Dict[str, any]:
        """Fetch a single URL and return the result."""
        try:
            response = requests.get(url, timeout=5)
            response.raise_for_status()
            return {
                "url": url,
                "status": response.status_code,
                "content_length": len(response.content),
            }
        except requests.exceptions.RequestException as e:
            return {"url": url, "error": str(e)}

    def scrape_urls(self, urls: List[str]) -> List[Dict[str, any]]:
        """
        Scrape multiple URLs concurrently.

        BUG: self.results is accessed from multiple threads without locking!
        This causes race conditions where results can be lost or corrupted.
        """
        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            futures = [executor.submit(self.fetch_url, url) for url in urls]

            for future in as_completed(futures):
                result = future.result()

                # RACE CONDITION: Multiple threads append to self.results simultaneously
                if "error" in result:
                    self.failed_urls.append(result["url"])  # RACE CONDITION
                else:
                    self.results.append(result)  # RACE CONDITION

        return self.results
```

**버그**: 여러 스레드가 락 없이 `self.results`와 `self.failed_urls`를 수정합니다!

Claude는 다음을 수행합니다.
1. 경쟁 상태를 찾아냅니다
2. 패턴을 `/memories/concurrency_patterns/thread_safety.md`에 저장합니다
3. 세션 2에서 이 동시성 패턴을 비동기 코드에 적용합니다

### 데모 개요

완전한 코드 리뷰 어시스턴트를 만들어 두었습니다. 구현은 `memory_demo/code_review_demo.py`에 있습니다.

**대화형 데모를 실행하려면:**
```bash
python memory_demo/code_review_demo.py
```

이 데모는 다음을 보여 줍니다.
1. **세션 1**: 버그가 있는 파이썬 코드 리뷰 → Claude가 패턴을 학습
2. **세션 2**: 비슷한 코드 리뷰(새 대화) → Claude가 패턴을 적용
3. **세션 3**: 긴 리뷰 세션 → 컨텍스트 편집이 감당 가능한 크기를 유지

## 7. 모범 사례와 보안 {#best-practices}

### 메모리 관리

**권장:**
- ✅ 대화 기록이 아니라 작업과 관련된 패턴을 저장하세요
- ✅ 명확한 디렉터리 구조로 정리하세요
- ✅ 설명적인 파일 이름을 쓰세요
- ✅ 주기적으로 메모리를 점검하고 정리하세요

**금지:**
- ❌ 민감 정보(비밀번호, API 키, 개인정보)를 저장하지 마세요
- ❌ 메모리가 무한정 커지게 두지 마세요
- ❌ 아무거나 무분별하게 저장하지 마세요

### 보안: 경로 탐색 공격 방어

**필수**: 디렉터리 탐색(path traversal) 공격을 막기 위해 항상 경로를 검증하세요. 구현은 `memory_tool.py`를 참고하세요.

### 보안: 메모리 오염

**⚠️ 중대한 위험**: 메모리 파일은 Claude의 컨텍스트로 다시 읽혀 들어오므로, 프롬프트 인젝션의 경로가 될 수 있습니다.

**완화 전략:**
1. **콘텐츠 정제**: 저장하기 전에 위험한 패턴을 걸러 내세요
2. **메모리 범위 격리**: 사용자별/프로젝트별로 분리하세요
3. **메모리 감사**: 모든 메모리 작업을 기록하고 검사하세요
4. **프롬프트 엔지니어링**: 메모리 안의 지시는 무시하도록 Claude에 지시하세요

전체 보안 구현은 `memory_tool.py`, 테스트는 `tests/`를 참고하세요.

## 마무리

### 해낸 것들

이 쿡북에서 여러분은 다음을 배웠습니다.
- ✅ 대화를 넘나드는 학습을 위해 **메모리 도구를 구현하기**(세션 1과 2에서 패턴 인식이 유지되는 것을 확인)
- ✅ 토큰 트리거와 보존 정책으로 **컨텍스트 편집을 설정하기**(세션 3에서 자동 정리를 확인)
- ✅ 경로 검증과 메모리 오염 방지를 포함한 **보안 모범 사례 적용하기**

### 이 패턴을 적용하기

**여러분의 프로젝트에서:**
1. 패턴을 담을 메모리 파일 하나로 시작하세요(예: `/memories/patterns.md`)
2. 프로덕션에서는 컨텍스트 편집 트리거를 30~40k 토큰으로 설정하세요
3. 상호 오염을 막기 위해 프로젝트별 메모리 격리를 구현하세요

**다른 활용처:**
- **고객 지원**: 사용자 선호와 자주 있는 문제의 해결책을 저장
- **리서치 어시스턴트**: 세션을 넘어 도메인 지식을 축적
- **데이터 분석**: 데이터셋 특성과 잘 통한 기법을 기억

### 다음 단계

- **프로덕션 배포**: GitHub PR 리뷰에 [claude-code-action](https://github.com/anthropics/claude-code-action)을 사용하세요
- **보안 강화**: `memory_tool.py`의 메모리 오염 완화 방안을 검토하세요
- **확장 사고**: 연산이 많은 작업을 위해 사고 관리를 살펴보세요

### 참고 자료

- [메모리 도구 문서](https://docs.claude.com/en/docs/agents-and-tools/tool-use/memory-tool)
- [Claude API 레퍼런스](https://docs.claude.com/en/api/messages)
- [지원](https://support.claude.com)

메모리와 컨텍스트 관리는 **베타** 단계입니다. 개선에 도움이 되도록 피드백을 보내 주세요!